# Export Winner & Reload Test

## Imports

In [ ]:
import json, hashlib, joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
from sklearn.model_selection import train_test_split

MODEL_VERSION = "1"
DATA_DIR = Path("../data/work")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / f"model_v{MODEL_VERSION}.joblib"
META_PATH  = MODELS_DIR / f"model_v{MODEL_VERSION}.json"

train_path = DATA_DIR / "train.csv"
test_path  = DATA_DIR / "test.csv"


df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

X_train = df_train.iloc[:, :-1].copy()
y_train = df_train.iloc[: , -1].astype(float).values
X_test  = df_test.iloc[:, :-1].copy()
y_test  = df_test.iloc[: , -1].astype(float).values

In [ ]:
def ensure_cols(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Colunas ausentes: {missing}")
    return cols


## Normalization and Pipelines

In [16]:
cat_cols  = ["address", "district", "type"]
num_cols  = ["bedrooms", "garage"]
area_col  = ["area"]
feature_cols = [c for c in (num_cols + area_col + cat_cols)]

def clip_upper(X, upper=None):
    import numpy as np
    return np.clip(X, None, upper)

if not isinstance(X_train, pd.DataFrame):
    X_train = pd.DataFrame(X_train, columns=feature_cols)
if not isinstance(X_test, pd.DataFrame):
    X_test  = pd.DataFrame(X_test,  columns=feature_cols)

missing = [c for c in feature_cols if c not in X_train.columns]
if missing:
    raise KeyError(f"As colunas esperadas não estão em X_train: {missing}")

for df in (X_train, X_test):
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.lower()
    if "area" in df.columns:
        df["area"] = pd.to_numeric(df["area"], errors="coerce")


p99_area = np.nanpercentile(X_train["area"].to_numpy(), 99)

Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

area_tree = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clip99", FunctionTransformer(
        clip_upper,
        kw_args={"upper": p99_area},
        feature_names_out="one-to-one"
    )),
])

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
])

ct_tree = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("area",      area_tree, area_col),  
        ("cat",       cat_pipe,  cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)


Xt_tree_tr = ct_tree.fit_transform(X_train)
Xt_tree_te = ct_tree.transform(X_test)

print("Tree CT -> train/test shapes:", Xt_tree_tr.shape, Xt_tree_te.shape)


Tree CT -> train/test shapes: (9325, 23) (2332, 23)


In [17]:
def eval_predictions(y_true, y_pred):
    return {
        "MAE":   mean_absolute_error(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2":    r2_score(y_true, y_pred),
    }

def eval_baseline_median(y_true, train_median):
    y_pred = np.full_like(y_true, fill_value=float(train_median), dtype=float)
    return eval_predictions(y_true, y_pred)

## Build and save the best model

In [18]:
y_train = pd.to_numeric(y_train, errors="coerce")

random_forest = RandomForestRegressor(random_state=42, n_jobs=-1)

pipe_forest_tunned = Pipeline(steps=[
    ("preprocess", ct_tree),       
    ("model", random_forest)
])


param_dist = {
    "model__n_estimators": [300, 600, 900],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": [1, 2, 4],
    "model__min_samples_split": [2, 5, 10],
    "model__max_features": ["sqrt", 0.5],
    "model__bootstrap": [True],
}


def build_best_pipe_from_params():
    rf = RandomizedSearchCV(
        estimator=pipe_forest_tunned,       
        param_distributions=param_dist,
        n_iter=20,
        scoring="neg_mean_absolute_error",
        cv=5,
        random_state=42,
        n_jobs=-1,
        error_score=np.nan
    )
    rf.fit(X_train, y_train)                 
    return rf.best_estimator_  

best_pipe = None
source = None

def file_size_ok(p: Path, min_bytes: int = 1000) -> bool:
    try:
        return p.exists() and p.stat().st_size >= min_bytes
    except Exception:
        return False

if file_size_ok(MODELS_DIR):
    try:
        best_pipe = joblib.load(MODELS_DIR)
        source = "loaded_existing_model_v1"
        print(f"Loaded existing model: {MODELS_DIR} ({MODELS_DIR.stat().st_size} bytes)")
    except Exception as e:
        print("Warning: failed to load existing model:", e)
        try:
            MODELS_DIR.unlink()
            print("Removed corrupted model file. Rebuilding from BEST_PARAMS…")
        except Exception as _:
            pass


if best_pipe is None:
    best_pipe = build_best_pipe_from_params()
    source = "rebuilt_from_best_params"
    model_path = MODELS_DIR / "best_forest_pipe.joblib"   
    joblib.dump(best_pipe, model_path)
    print("Saved rebuilt model to:", model_path)

source

Saved rebuilt model to: ..\models\best_forest_pipe.joblib


'rebuilt_from_best_params'

In [ ]:
if not MODEL_PATH.exists():
    joblib.dump(best_pipe, MODEL_PATH)
    print("Saved:", MODEL_PATH)
else:
    print("Model already existed on disk — keeping it:", MODEL_PATH)

y_pred_test = best_pipe.predict(X_test)
test_mae   = float(mean_absolute_error(y_test, y_pred_test))
test_medae = float(median_absolute_error(y_test, y_pred_test))
test_r2    = float(r2_score(y_test, y_pred_test))

train_hash = hashlib.sha256((DATA_DIR / "train.csv").read_bytes()).hexdigest()

meta = {
    "model_version": MODEL_VERSION,
    "source": source,
    "dataset": {"train_file": str(train_path), "train_hash": train_hash},
    "metrics_test": {"MAE": test_mae, "MedAE": test_medae, "R2": test_r2},
    "framework": "scikit-learn",
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)
print("Saved:", META_PATH)
meta

Saved: ..\models\model_v1.joblib
Saved: ..\models\model_v1.json


{'model_version': '1',
 'source': 'rebuilt_from_best_params',
 'dataset': {'train_file': '..\\data\\work\\train.csv',
  'train_hash': 'b1b4c25a1cab3fdf8de23cd4dc5d32d62d30c74cb2ddce6b5f3683dc6fb05345'},
 'metrics_test': {'MAE': 1233.6758867962042,
  'MedAE': 752.9690466213208,
  'R2': 0.6853433884252562},
 'framework': 'scikit-learn'}

## Reload test (contract): ensure we can load and predict the same way

In [20]:
reloaded = joblib.load(MODEL_PATH)
y_pred_reload = reloaded.predict(X_test)

diff = np.abs(y_pred_reload - y_pred_test).mean()
print(f"Mean |Δ| between original and reloaded predictions: {diff:.8f}")
assert diff < 1e-9, "Reloaded predictions differ more than expected."

print("✅ Reload test passed.")

Mean |Δ| between original and reloaded predictions: 0.00000000
✅ Reload test passed.


## Conformal calibration

In [ ]:
assert "best_pipe" in globals(), "best_pipe not found. Run model selection cell first."
assert "X_train" in globals() and "y_train" in globals(), "X_train/y_train not found."

MODEL_VERSION = globals().get("MODEL_VERSION", "1")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / f"model_v{MODEL_VERSION}.joblib"
META_PATH  = MODELS_DIR / f"model_v{MODEL_VERSION}.json"

X_tr, X_cal, y_tr, y_cal = train_test_split(
    X_train, y_train, test_size=0.10, random_state=42
)

best_pipe = best_pipe.fit(X_tr, y_tr)

cal_pred = best_pipe.predict(X_cal)
resid = np.abs(y_cal - cal_pred)

qhat_90 = float(np.quantile(resid, 0.90))

meta = {}
if META_PATH.exists():
    try:
        meta = json.loads(META_PATH.read_text())
    except Exception:
        meta = {}

meta["model_version"] = str(MODEL_VERSION)
meta.setdefault("uncertainty", {})["qhat_mae_90"] = qhat_90

META_PATH.write_text(json.dumps(meta, indent=2))

best_pipe = best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, MODEL_PATH)

print(f"[Conformal] qhat_90 = {qhat_90:.4f} saved into {META_PATH}")
print(f"[Export] Model re-saved at: {MODEL_PATH}")


[Conformal] qhat_90 = 2699.2500 saved into models\model_v1.json
[Export] Model re-saved at: models\model_v1.joblib
